In [ ]:
pip install arch

# Study: Portfolio VaR & ES
## Quantitative Financial Risk Management


### Portfolio Composition

| Asset | Ticker | Sector | Currency | Type |
|-------|--------|--------|----------|------|
| Maersk | MAERSK-B.CO | Shipping | DKK | Stock |
| Agnico Eagle Mines | AEM | Gold Mining | USD | Stock |
| ASML | ASML | Semiconductors | EUR | Stock |
| S&P 500 | ^GSPC | — | USD | Index |
| Euro Stoxx 50 | ^STOXX50E | — | EUR | Index |

We had to make things interesting and we know these stock are interesting since they are very sharp indicatoors for certain sectors. Three stocks across different sectors (shipping, mining, tech) and three currencies (DKK, USD, EUR) ensure exposure to equity, FX, and sector-specific risk. Gold mining acts as a potential safe-haven hedge, shipping is cyclical/trade-sensitive, and ASML represents tech/growth. Moreover we chose more standard indices like the  s&P500 and euro stoxx since we have both USD denominated stocks as more EU stocks. 

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 20)

tickers = ['MAERSK-B.CO', 'AEM', 'ASML.AS', '^GSPC', '^STOXX50E', 'EURUSD=X', 'DKKUSD=X']

raw = yf.download(tickers, start='2016-01-01', end='2026-03-31', auto_adjust=True)
prices = raw['Close'].copy()

prices = prices.rename(columns={
    'MAERSK-B.CO': 'Maersk',
    'AEM': 'Agnico_Eagle',
    'ASML.AS': 'ASML',
    '^GSPC': 'SP500',
    '^STOXX50E': 'EuroStoxx50',
    'EURUSD=X': 'EURUSD',
    'DKKUSD=X': 'DKKUSD'
})

print(prices.head())


print("Missing values per column:")
print(prices.isna().sum())
print(f"\nTotal rows: {len(prices)}")
print(f"Rows with any NaN: {prices.isna().any(axis=1).sum()}")

"we first check if we can forward-fill single-day gaps (e.g. different holiday calendars), then drop any remaining rows with NaN if there are to many in that period (we only fill 1 consecutive nan to nott generate to much data). "
prices_clean = prices.ffill(limit=1)  
dropped = prices_clean.isna().any(axis=1).sum()
prices_clean = prices_clean.dropna()

print(f"Rows dropped after forward-fill(1): {dropped}")
print(f"Final shape: {prices_clean.shape}")
print(f"Final date range: {prices_clean.index.min().date()} to {prices_clean.index.max().date()}")

# Convert all asset prices to USD using historical FX rates
prices_usd = pd.DataFrame(index=prices_clean.index)

# Already in USD
prices_usd['Agnico_Eagle'] = prices_clean['Agnico_Eagle']
prices_usd['SP500'] = prices_clean['SP500']

# EUR assets -> USD
prices_usd['ASML'] = prices_clean['ASML'] * prices_clean['EURUSD']
prices_usd['EuroStoxx50'] = prices_clean['EuroStoxx50'] * prices_clean['EURUSD']

# DKK asset -> USD
prices_usd['Maersk'] = prices_clean['Maersk'] * prices_clean['DKKUSD']

# Add DTB3 cash component
dtb3 = pd.read_csv("DTB3.csv")
dtb3.columns = [c.strip() for c in dtb3.columns]

if 'DATE' in dtb3.columns:
    dtb3['DATE'] = pd.to_datetime(dtb3['DATE'])
    dtb3 = dtb3.set_index('DATE')
elif 'observation_date' in dtb3.columns:
    dtb3['observation_date'] = pd.to_datetime(dtb3['observation_date'])
    dtb3 = dtb3.set_index('observation_date')
else:
    raise ValueError("No valid date column found in DTB3.csv")

dtb3['DTB3'] = pd.to_numeric(dtb3['DTB3'], errors='coerce')
dtb3 = dtb3.reindex(prices_usd.index).ffill()

# annual rate-> approximate daily return > was doubting for a second if we should use 364 here? 
dtb3['daily_cash_return'] = (dtb3['DTB3'] / 100) / 252
#cummulate the returns because the loan should grow over time with the discount rate
dtb3['USD_Cash'] = 100 * (1 + dtb3['daily_cash_return']).cumprod()

# Final dataset
prices_clean = prices_usd.copy()
prices_clean['USD_Cash'] = dtb3['USD_Cash']

print("Converted everything to USD and added cash component.")
prices_clean.head()

log_returns = np.log(prices_clean / prices_clean.shift(1)).dropna()
print(f"Returns shape: {log_returns.shape}")
log_returns.describe().round(6)



#normalized prices
normalized = 100 * prices_clean / prices_clean.iloc[0]

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

normalized.plot(ax=axes[0], title='Normalized Prices (base=100)')
axes[0].legend(loc='upper left')
axes[0].set_ylabel('Price (indexed)')

log_returns.plot(ax=axes[1], alpha=0.5, title='Daily Log Returns')
axes[1].legend(loc='upper left')
axes[1].set_ylabel('Log return')

plt.tight_layout()
plt.show()

prices_clean.to_csv('portfolio_prices_usd.csv')
log_returns.to_csv('portfolio_log_returns_usd.csv')
prices_clean.to_excel('portfolio_prices_usd.xlsx')

print('Saved portfolio_prices_usd.csv, portfolio_log_returns_usd.csv, portfolio_prices_usd.xlsx')

summary = pd.DataFrame({'Ann. Mean (%)': log_returns.mean() * 252 * 100,'Ann. Vol (%)': log_returns.std() * np.sqrt(252) * 100,'Skewness': log_returns.skew(),'Kurtosis': log_returns.kurtosis(),'Min (%)': log_returns.min() * 100,'Max (%)': log_returns.max() * 100,}).round(3)
print("Correlation matrix:")
display(log_returns.corr().round(3))
print("\nSummary statistics:")
display(summary)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from arch import arch_model



df_logret = pd.read_csv("portfolio_log_returns_usd.csv", index_col=0, parse_dates=True)
lst_all   = df_logret.columns.tolist()

print(f"Sample: {df_logret.index[0].date()} to {df_logret.index[-1].date()}, "
      f"{len(df_logret)} observations\n")


# 2. PARAMETERS

flt_alpha     = 0.01
int_ewma_span = 60
lst_t_dfs     = [3, 4, 5, 6]

# 3. METHODS

def calc_var_es_normal(arr_returns, flt_alpha):
    flt_mu    = np.mean(arr_returns)
    flt_sigma = np.std(arr_returns, ddof=1)
    flt_z     = stats.norm.ppf(flt_alpha)
    flt_var = -(flt_mu + flt_sigma * flt_z)
    flt_es  = -(flt_mu - flt_sigma * stats.norm.pdf(flt_z) / flt_alpha)
    return flt_var, flt_es


def calc_var_es_student_t(arr_returns, flt_alpha, int_df):
    flt_mu    = np.mean(arr_returns)
    flt_sigma = np.std(arr_returns, ddof=1)
    flt_scale = flt_sigma * np.sqrt((int_df - 2) / int_df)
    flt_t_q   = stats.t.ppf(flt_alpha, df=int_df)
    flt_var = -(flt_mu + flt_scale * flt_t_q)
    flt_es  = -(flt_mu - flt_scale * (
        stats.t.pdf(flt_t_q, df=int_df) / flt_alpha
        * (int_df + flt_t_q**2) / (int_df - 1)
    ))
    return flt_var, flt_es


def calc_var_es_historical(arr_returns, flt_alpha):
    flt_var = -np.percentile(arr_returns, flt_alpha * 100)
    flt_es  = -np.mean(arr_returns[arr_returns <= -flt_var])
    return flt_var, flt_es


def calc_var_es_garch(arr_returns, flt_alpha):
    mdl_garch = arch_model(arr_returns * 100, vol="Garch", p=1, q=1,
                           mean="Constant", dist="normal")
    res_garch = mdl_garch.fit(disp="off")
    flt_mu_pct    = res_garch.params["mu"]
    flt_sigma_pct = np.sqrt(res_garch.forecast(horizon=1).variance.values[-1, 0])
    flt_z   = stats.norm.ppf(flt_alpha)
    flt_var = -(flt_mu_pct + flt_sigma_pct * flt_z) / 100
    flt_es  = -(flt_mu_pct - flt_sigma_pct * stats.norm.pdf(flt_z) / flt_alpha) / 100
    return flt_var, flt_es, res_garch


def calc_var_es_fhs_ewma(arr_returns, flt_alpha, int_span):
    ser_returns  = pd.Series(arr_returns)
    flt_mu       = np.mean(arr_returns)
    ser_ewma_var = ser_returns.ewm(span=int_span).var()
    arr_std_resid = ((arr_returns[1:] - flt_mu) / np.sqrt(ser_ewma_var.values[:-1]))
    arr_std_resid = arr_std_resid[np.isfinite(arr_std_resid)]
    flt_sigma_last = np.sqrt(ser_ewma_var.values[-1])
    flt_q_resid    = np.percentile(arr_std_resid, flt_alpha * 100)
    flt_var = -(flt_mu + flt_sigma_last * flt_q_resid)
    arr_tail = arr_std_resid[arr_std_resid <= flt_q_resid]
    flt_es   = -(flt_mu + flt_sigma_last * np.mean(arr_tail))
    return flt_var, flt_es

# 4. COMPUTE VaR & ES FOR EACH ASSET

dict_results = {}

for str_asset in lst_all:
    arr_ret = df_logret[str_asset].values
    dict_asset = {}

    flt_var, flt_es = calc_var_es_normal(arr_ret, flt_alpha)
    dict_asset["Normal"] = {"VaR": flt_var, "ES": flt_es}

    for int_df in lst_t_dfs:
        flt_var, flt_es = calc_var_es_student_t(arr_ret, flt_alpha, int_df)
        dict_asset[f"t(df={int_df})"] = {"VaR": flt_var, "ES": flt_es}

    flt_var, flt_es = calc_var_es_historical(arr_ret, flt_alpha)
    dict_asset["HistSim"] = {"VaR": flt_var, "ES": flt_es}

    flt_var, flt_es, _ = calc_var_es_garch(arr_ret, flt_alpha)
    dict_asset["GARCH"] = {"VaR": flt_var, "ES": flt_es}

    flt_var, flt_es = calc_var_es_fhs_ewma(arr_ret, flt_alpha, int_ewma_span)
    dict_asset["FHS-EWMA"] = {"VaR": flt_var, "ES": flt_es}

    dict_results[str_asset] = dict_asset

# 5. RESULTS TABLE

lst_rows = []
for str_asset, dict_methods in dict_results.items():
    for str_method, dict_vals in dict_methods.items():
        lst_rows.append({
            "Asset":  str_asset,
            "Method": str_method,
            "VaR":    dict_vals["VaR"],
            "ES":     dict_vals["ES"],
        })

df_results = pd.DataFrame(lst_rows)
df_results["VaR"] = df_results["VaR"].map(lambda x: f"{x:.6f}")
df_results["ES"]  = df_results["ES"].map(lambda x: f"{x:.6f}")

print("=" * 60)
print(f"  Univariate 1-day VaR & ES  (alpha = {flt_alpha})")
print("=" * 60)

for str_asset in lst_all:
    df_sub = df_results[df_results["Asset"] == str_asset]
    print(f"\n--- {str_asset} ---")
    print(df_sub[["Method", "VaR", "ES"]].to_string(index=False))

# 6. QQ-PLOTS

fig_qq, arr_axes = plt.subplots(len(lst_all), len(lst_t_dfs),
                                figsize=(16, 3 * len(lst_all)),
                                squeeze=False)

for int_i, str_asset in enumerate(lst_all):
    arr_ret = df_logret[str_asset].values
    flt_mu  = np.mean(arr_ret)
    flt_sig = np.std(arr_ret, ddof=1)
    arr_standardised = (arr_ret - flt_mu) / flt_sig

    for int_j, int_df in enumerate(lst_t_dfs):
        ax_cur = arr_axes[int_i, int_j]
        stats.probplot(arr_standardised, dist=stats.t, sparams=(int_df,),
                       plot=ax_cur)
        ax_cur.set_title(f"{str_asset}  t(df={int_df})", fontsize=9)
        ax_cur.get_lines()[0].set_markersize(2)

fig_qq.tight_layout()
fig_qq.savefig("qq_plots_student_t.png", dpi=150)
plt.show()

print("\nQQ plots saved to qq_plots_student_t.png")
print("Done.")


# Portfolio Weights & VaR,ES for Portfolio & Backtests

- we need to cchoose and fix portfolio weights
    - equal weights
    - risk parity weights
    - MV Portfolio optimisation
- Compute portfolio-level VaR & ES using 5 methods for 1 day, 5day and 10day ahead
- Stress Testing 
    - • Equity index values or stock prices changing by +/- 20% and +/- 40% of the current values
    - • Currencies moving by +/- 10% for major currencies and +/- 20% for other currencies
    - • Commodity prices changing by +/- 20% and +/- 40% of the current values
    - • Interest rates shifting by +/- 2% and +/- 3%

## We check for stationarity at the monthly,quarterly,half year and yearly level over the 10years to create portfolio weights that are stationary.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_pacf
from statsmodels.graphics.tsaplots import plot_acf

def analyze_stationarity_across_frequencies(df_daily_log_returns):
    """
    Evaluates the ADF test and plots PACF for log returns across multiple temporal frequencies.
    
    Parameters:
    df_daily_log_returns: DataFrame of daily log returns with a DatetimeIndex.
    """
    # Define the mapping of pandas offset aliases to human-readable strings
    frequencies = {
        'ME': 'Monthly', 
        'QE': 'Quarterly', 
        '6ME': 'Half-Yearly', 
        'YE': 'Yearly'
    }
    
    for freq_code, freq_name in frequencies.items():
        print(f"\n{'='*50}")
        print(f"FREQUENCY: {freq_name} (Resample Code: {freq_code})")
        print(f"{'='*50}")
        
        # 1. Temporally aggregate log returns via summation
        df_resampled = df_daily_log_returns.resample(freq_code).sum().dropna()
        
        T, N = df_resampled.shape
        print(f"Sample Size (T) at this frequency: {T}\n")
        
        # 2. Run ADF Test for each asset
        print(f"{'Asset':<15} | {'ADF Stat':<10} | {'p-value':<10} | {'Stationary (p<0.05)'}")
        print("-" * 60)
        
        for col in df_resampled.columns:
            series = df_resampled[col]
            
            # Skip ADF if T is dangerously small (e.g., Yearly with T=10), 
            # though we allow it to run here to observe the degradation of test power.
            try:
                # autolag='AIC' dynamically selects the optimal number of lagged differences (p)
                adf_result = adfuller(series, autolag='AIC')
                stat, p_value = adf_result[0], adf_result[1]
                is_stat = "Yes" if p_value < 0.05 else "NO"
                print(f"{col:<15} | {stat:<10.4f} | {p_value:<10.4f} | {is_stat}")
            except ValueError as e:
                print(f"{col:<15} | Insufficient data for ADF test.")
        
        # 3. Plot PACF for visual inspection of orthogonal projections
        # We limit the number of lags to T/2 to avoid estimation instability
        nlags = min(10, int(T / 2) - 1) 
        
        if nlags > 1:
            fig, axes = plt.subplots(1, N, figsize=(4 * N, 3), squeeze=False)
            fig.suptitle(f"PACF Projections - {freq_name} Frequency", y=1.05)
            
            for i, col in enumerate(df_resampled.columns):
                # The method='ywm' uses the Yule-Walker equations to solve the projection system
                plot_pacf(df_resampled[col], lags=nlags, ax=axes[0, i], title=col, method='ywm')
            
            plt.tight_layout()
            plt.show()
        else:
            print(f"\n[!] Not enough observations to compute a meaningful PACF at {freq_name} frequency.")

# Example execution:
# Note: Ensure df_logret has a proper DatetimeIndex before running.
analyze_stationarity_across_frequencies(log_returns)

## Portfolio Weights

In [ ]:
# USD_Cash is excluded from the CCC-GARCH estimation:
# its return series is near-deterministic, so GARCH fitting is numerically unstable
# and does not add meaningful information. We therefore model CCC-GARCH only for the
# risky assets and add the cash leg separately at the portfolio aggregation stage.


def calc_var_es_ccc_garch(df_logret, ser_weights, flt_alpha=0.01):
    """
    CCC-GARCH(1,1) portfolio VaR and ES under conditional normality.

    Inputs
    ------
    df_logret : pd.DataFrame
        DataFrame of aligned asset log returns.
    ser_weights : pd.Series
        Portfolio weights indexed by the same asset names as df_logret.columns.
    flt_alpha : float
        Tail probability, e.g. 0.01 for 99% VaR/ES.

    Returns
    -------
    dict
        Dictionary with portfolio VaR, ES, mean forecast, volatility forecast,
        conditional covariance forecast, correlation matrix, and fitted models.
    """

    lst_assets = df_logret.columns.tolist()

    ser_weights = ser_weights.reindex(lst_assets)
    if ser_weights.isna().any():
        raise ValueError("ser_weights must contain all assets in df_logret.columns")

    if not np.isclose(ser_weights.sum(), 1.0):
        raise ValueError("Portfolio weights must sum to 1")
    # cash removal 
    lst_risky = [c for c in lst_assets if c != "USD_Cash"]

    dict_mu = {}
    dict_sigma_next = {}
    dict_std_resid = {}
    dict_models = {}

    #for str_asset in lst_assets:
    for str_asset in lst_risky:
        arr_ret = df_logret[str_asset].dropna().values
        #flt_std = np.std(arr_ret, ddof=1)

        mdl_garch = arch_model(
            arr_ret * 100,
            mean="Constant",
            vol="Garch",
            p=1,
            q=1,
            dist="normal",
            rescale=False
        )

        res_garch = mdl_garch.fit(disp="off")

        flt_mu_hat = res_garch.params["mu"] / 100
        arr_sigma_t = res_garch.conditional_volatility / 100
        flt_sigma_next = np.sqrt(
            res_garch.forecast(horizon=1, reindex=False).variance.values[-1, 0]
        ) / 100

        arr_std_resid = (arr_ret - flt_mu_hat) / arr_sigma_t
        arr_std_resid = np.where(np.isfinite(arr_std_resid), arr_std_resid, np.nan)

        dict_mu[str_asset] = flt_mu_hat
        dict_sigma_next[str_asset] = flt_sigma_next
        dict_std_resid[str_asset] = pd.Series(arr_std_resid, index=df_logret.index)
        dict_models[str_asset] = res_garch

    df_std_resid = pd.DataFrame(dict_std_resid).dropna()
    mat_R = df_std_resid.corr().values

    #vec_sigma_next = np.array([dict_sigma_next[str_asset] for str_asset in lst_assets])
    vec_sigma_next = np.array([dict_sigma_next[str_asset] for str_asset in lst_risky])
    mat_D_next = np.diag(vec_sigma_next)
    mat_Sigma_next = mat_D_next @ mat_R @ mat_D_next

    #vec_mu = np.array([dict_mu[str_asset] for str_asset in lst_assets])
    #vec_w = ser_weights.values
    vec_mu_risky = np.array([dict_mu[str_asset] for str_asset in lst_risky])
    vec_w_risky = ser_weights[lst_risky].values

    # Cash leg added separately
    flt_mu_cash = df_logret["USD_Cash"].mean()
    flt_w_cash = ser_weights["USD_Cash"]

    #flt_mu_p = float(vec_w @ vec_mu)
    #flt_sigma_p = float(np.sqrt(vec_w @ mat_Sigma_next @ vec_w))
    flt_mu_p = float(vec_w_risky @ vec_mu_risky + flt_w_cash * flt_mu_cash)
    flt_sigma_p = float(np.sqrt(vec_w_risky @ mat_Sigma_next @ vec_w_risky))

    flt_z = stats.norm.ppf(flt_alpha)

    flt_var_p = -(flt_mu_p + flt_sigma_p * flt_z)
    flt_es_p = -(flt_mu_p - flt_sigma_p * stats.norm.pdf(flt_z) / flt_alpha)

    return {
        "VaR": flt_var_p,
        "ES": flt_es_p,
        "mu_portfolio": flt_mu_p,
        "sigma_portfolio": flt_sigma_p,
        "Sigma_next": pd.DataFrame(mat_Sigma_next, index=lst_risky, columns=lst_risky),
        "R": pd.DataFrame(mat_R, index=lst_risky, columns=lst_risky),
        "mu_assets": pd.Series(dict_mu),
        "sigma_next_assets": pd.Series(dict_sigma_next),
        "std_residuals": df_std_resid,
        "models": dict_models,
    }


def calc_ewma_sigma(arr_returns, flt_lambda=0.94, flt_mu=None):
    """
    Standard RiskMetrics-style EWMA volatility filter.

    sigma_t^2 = lambda * sigma_{t-1}^2 + (1-lambda) * eps_{t-1}^2

    Returns
    -------
    arr_sigma_t : np.ndarray
        Conditional sigmas aligned with arr_returns.
    flt_sigma_next : float
        One-step-ahead sigma forecast.
    flt_mu : float
        Mean used in the filter.
    """

    arr_returns = np.asarray(arr_returns, dtype=float)

    if flt_mu is None:
        flt_mu = np.mean(arr_returns)

    arr_eps = arr_returns - flt_mu
    int_n = len(arr_eps)

    arr_sigma2_t = np.empty(int_n)
    arr_sigma2_t[0] = np.var(arr_eps, ddof=1)

    for int_t in range(1, int_n):
        arr_sigma2_t[int_t] = (
            flt_lambda * arr_sigma2_t[int_t - 1]
            + (1 - flt_lambda) * arr_eps[int_t - 1] ** 2
        )

    flt_sigma2_next = (
        flt_lambda * arr_sigma2_t[-1]
        + (1 - flt_lambda) * arr_eps[-1] ** 2
    )

    arr_sigma_t = np.sqrt(arr_sigma2_t)
    flt_sigma_next = np.sqrt(flt_sigma2_next)

    return arr_sigma_t, flt_sigma_next, flt_mu

def calc_portfolio_var_es_fhs_ewma(df_logret, ser_weights, flt_alpha=0.01, flt_lambda=0.94):
    """
    Portfolio 1-day VaR and ES using multivariate Filtered Historical Simulation
    with an EWMA volatility filter for each asset.

    Method:
    1. Filter each asset return series with EWMA
    2. Construct standardized residual vectors z_t
    3. Keep the historical dependence structure across assets
    4. Rescale each historical residual vector by the current EWMA volatility forecast
    5. Form portfolio scenario returns and estimate VaR / ES empirically
    """

    lst_assets = df_logret.columns.tolist()

    ser_weights = ser_weights.reindex(lst_assets)
    if ser_weights.isna().any():
        raise ValueError("ser_weights must contain all assets in df_logret.columns")

    if not np.isclose(ser_weights.sum(), 1.0):
        raise ValueError("Portfolio weights must sum to 1")

    dict_mu = {}
    dict_sigma_t = {}
    dict_sigma_next = {}
    dict_z = {}

    for str_asset in lst_assets:
        arr_ret = df_logret[str_asset].dropna().values

        arr_sigma_t, flt_sigma_next, flt_mu = calc_ewma_sigma(
            arr_returns=arr_ret,
            flt_lambda=flt_lambda,
            flt_mu=np.mean(arr_ret)
        )

        # Numerical floor for very low-volatility series such as USD_Cash
        arr_sigma_t = np.sqrt(np.maximum(arr_sigma_t**2, 1e-12))
        flt_sigma_next = np.sqrt(max(flt_sigma_next**2, 1e-12))

        arr_z = (arr_ret - flt_mu) / arr_sigma_t
        arr_z = np.where(np.isfinite(arr_z), arr_z, np.nan)

        dict_mu[str_asset] = flt_mu
        dict_sigma_t[str_asset] = pd.Series(arr_sigma_t, index=df_logret.index)
        dict_sigma_next[str_asset] = flt_sigma_next
        dict_z[str_asset] = pd.Series(arr_z, index=df_logret.index)

    df_z = pd.DataFrame(dict_z).dropna()

    vec_mu = np.array([dict_mu[str_asset] for str_asset in lst_assets])
    vec_sigma_next = np.array([dict_sigma_next[str_asset] for str_asset in lst_assets])
    vec_w = ser_weights.values

    # Historical filtered scenarios for next-day asset returns
    mat_ret_scen = vec_mu + df_z.values * vec_sigma_next

    # Portfolio scenario returns
    arr_port_scen = mat_ret_scen @ vec_w

    # Empirical VaR and ES from simulated portfolio return scenarios
    flt_q = np.quantile(arr_port_scen, flt_alpha)
    flt_var = -flt_q
    flt_es = -np.mean(arr_port_scen[arr_port_scen <= flt_q])

    return {
        "VaR": flt_var,
        "ES": flt_es,
        "scenario_returns_portfolio": arr_port_scen,
        "std_residuals": df_z,
        "mu_assets": pd.Series(dict_mu),
        "sigma_next_assets": pd.Series(dict_sigma_next),
    }



In [ ]:
# Get weights for different assets using the different methods
def compute_advanced_rolling_weights(df_returns, risky_assets, cash_asset, window=250, cash_weight=0.20, gamma=5.0):
    """
    Computes rolling Equal Weight, Risk Parity, and Rank-Injected Mean-Variance.
    Include risk aversion (gamma) to control the tilt towards the asset ranking signal in the rank-variance optimization.
    """
    # 1. State Space Setup
    T = len(df_returns)
    N = len(risky_assets)
    ones = np.ones(N)
    risky_budget = 1.0 - cash_weight
    
    # Extract raw numpy arrays for speed and linear algebra
    R_risky = df_returns[risky_assets].values
    
    # Output containers
    w_eq_mat = np.zeros((T, N))
    w_rp_mat = np.zeros((T, N))
    w_rmv_mat = np.zeros((T, N))
    
    for t in range(window, T):
        # The local filtration (rolling window)
        window_returns = R_risky[t-window:t, :]
        
        # ---------------------------------------------------------
        # 1. EQUAL WEIGHTS (Risky Partition)
        # ---------------------------------------------------------
        w_eq_mat[t, :] = (ones / N) * risky_budget
        
        # ---------------------------------------------------------
        # 2. RISK PARITY (Inverse Volatility)
        # ---------------------------------------------------------
        # sigma = sample standard deviation
        vols = np.std(window_returns, axis=0, ddof=1)
        inv_vol = 1.0 / vols
        w_rp_mat[t, :] = (inv_vol / np.sum(inv_vol)) * risky_budget
        
        # ---------------------------------------------------------
        # 3. RANK-INJECTED MEAN VARIANCE (Two-Fund Separation)
        # ---------------------------------------------------------
        # Calculate covariance matrix of the structural assets
        Sigma = np.cov(window_returns, rowvar=False, ddof=1)
        Sigma_inv = np.linalg.inv(Sigma)
        
        # Extract the ordinal rank signal
        cum_rets = np.sum(window_returns, axis=0)
        temp = cum_rets.argsort()
        ranks = np.empty_like(temp)
        ranks[temp] = np.arange(N) + 1  # Ranks from 1 (worst) to N (best)
        
        #  Standardize and Scale the Signal using Z-score for the ranks
        z_scores = (ranks - np.mean(ranks)) / np.std(ranks)
        mu = z_scores * 0.0005  # Scale the z-scores of the ranks to daily returns
        
        # Matrix Algebra for the Two-Fund Separation
        A = ones.T @ Sigma_inv @ ones
        B = mu.T @ Sigma_inv @ ones
        
        # Fund 1: Global Minimum Variance
        w_gmv = (Sigma_inv @ ones) / A
        
        # Fund 2: Zero-Cost Rank Tilt
        w_spec = (Sigma_inv @ mu) - (B / A) * (Sigma_inv @ ones)
        
        # Combine and apply the risky budget constraint
        # Using gamma=2.0 as a baseline risk aversion penalty
        w_risky_opt = w_gmv + (1.0 / 2.0) * w_spec
        w_rmv_mat[t, :] = w_risky_opt * risky_budget
    # ---------------------------------------------------------
    # Reassemble into DataFrames and append Cash
    # ---------------------------------------------------------
    dates = df_returns.index
    
    # Helper to construct df
    def make_df(w_mat):
        df = pd.DataFrame(w_mat, index=dates, columns=risky_assets)
        df[cash_asset] = cash_weight
        return df.iloc[window:] # Drop the burn-in period
    
    df_eq = make_df(w_eq_mat)
    df_rp = make_df(w_rp_mat)
    df_rmv = make_df(w_rmv_mat)
    
    # Heuristic Check: Verify rows sum to 1
    assert np.allclose(df_rmv.sum(axis=1), 1.0), "RMV Budget broken!"
    
    return df_eq, df_rp, df_rmv

# --- Example Execution ---
risky = ['Maersk', 'Agnico_Eagle', 'ASML', 'SP500', 'EuroStoxx50']
cash = 'USD_Cash'
df_eq, df_rp, df_rmv = compute_advanced_rolling_weights(log_returns, risky, cash, window=250, cash_weight=0.20, gamma=10.0)

print("Equal Weighting :")
print(df_eq.tail().round(4))

print("Risk Parity :")
print(df_rp.tail().round(4))

print("Rank-Injected Mean Variance :")
df_rmv.tail().round(4)

In [ ]:

weight_dist , axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].set_title("Equal Weighting Distribution")
df_eq['EuroStoxx50'].hist(bins=20, alpha=0.7, ax=axes[0])
axes[0].set_xlabel("Weight")

df_rmv['EuroStoxx50'].hist(bins=20, alpha=0.7, ax=axes[1])
axes[1].set_xlabel("Weight")
axes[1].set_title("Rank-Injected Mean Variance Distribution")

df_rp['EuroStoxx50'].hist(bins=20, alpha=0.7, ax=axes[2])
axes[2].set_xlabel("Weight")
axes[2].set_title("Risk Parity Distribution")
plt.tight_layout()
plt.show()

## Portfilio level VaR & ES + backtest
- For historical simulation check stationarity at different windows to use that as the window.

## Backtests

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from joblib import Parallel, delayed
from tqdm import tqdm # For a progress bar


# --- 1. The Single Day Evaluator ---
def evaluate_single_day(t, df_returns, df_weights, window, alpha):
    """
    Evaluates all 5 Risk Models for a single out-of-sample day.
    Designed to be run in parallel across CPU cores.
    """
    # Extract the data for the lookback window
    R_window = df_returns.iloc[t-window : t]
    w_current = df_weights.iloc[t-1]
    
    # --- Models 1, 2, 3: Static Models ---
    # (Using the logic we built previously)
    arr_rp = R_window.values @ w_current.values
    
    # 1. HS
    var_hs = -np.percentile(arr_rp, alpha * 100)
    tail_hs = arr_rp[arr_rp <= -var_hs]
    es_hs = -np.mean(tail_hs) if len(tail_hs) > 0 else var_hs
    
    # 2 Normal
    mu_p, sig_p = np.mean(arr_rp), np.std(arr_rp, ddof=1)
    z_alpha = stats.norm.ppf(alpha)
    var_norm = -(mu_p + sig_p * z_alpha)
    es_norm = -(mu_p - sig_p * (stats.norm.pdf(z_alpha) / alpha))

    # 3. Student-t (df=5) 
    nu = 5
    t_alpha = stats.t.ppf(alpha, df=nu)
    var_t = -(mu_p + t_alpha * sig_p * np.sqrt((nu - 2)/nu))
    t_pdf_alpha = stats.t.pdf(t_alpha, df=nu)
    es_t = -(mu_p - sig_p * np.sqrt((nu-2)/nu) * (t_pdf_alpha / alpha) * ((nu + t_alpha**2) / (nu - 1)))
    
    # 4. CCC-GARCH
    try:
        dict_ccc = calc_var_es_ccc_garch(R_window, w_current, flt_alpha=alpha)
        var_garch, es_garch = dict_ccc['VaR'], dict_ccc['ES']
    except:
        var_garch, es_garch = np.nan, np.nan # Failsafe if GARCH optimizer fails to converge

    # 5. FHS-EWMA
    dict_fhs = calc_portfolio_var_es_fhs_ewma(R_window, w_current, flt_alpha=alpha, flt_lambda=0.94)
    var_ewma, es_ewma = dict_fhs['VaR'], dict_fhs['ES']
    
    # Actual Return for day t
    actual_return = np.dot(df_returns.iloc[t].values, w_current.values)
    
    return {
        'Date': df_returns.index[t],
        'Actual_Return': actual_return,
        'VaR_HS': var_hs, 'ES_HS': es_hs,
        'VaR_Norm': var_norm, 'ES_Norm': es_norm,
        'VaR_t': var_t, 'ES_t': es_t,
        'VaR_GARCH': var_garch, 'ES_GARCH': es_garch,
        'VaR_EWMA': var_ewma, 'ES_EWMA': es_ewma
    }

# --- 2. The Parallel Executor ---
def run_parallel_backtest(df_returns, df_weights, window=250, alpha=0.01, n_jobs=-1):
    """
    Distributes the 2500-day rolling backtest across all available CPU cores.
    """
    df_returns = df_returns.loc[df_weights.index]
    T = len(df_returns)
    
    print(f"Backtest for {T - window} days on {n_jobs if n_jobs != -1 else 'ALL'} cores...")
    
    # Runs the loop simultaneously across your CPU cores
    results = Parallel(n_jobs=n_jobs)(
        delayed(evaluate_single_day)(t, df_returns, df_weights, window, alpha) 
        for t in tqdm(range(window, T))
    )
    
    df_results = pd.DataFrame(results).set_index('Date')
    return df_results




def plot_portfolio_qq_degrees_of_freedom(df_backtest):
    """
    Standardizes the out-of-sample portfolio returns and tests them 
    against t-distributions with varying degrees of freedom.
    """
    # Extract actual out-of-sample portfolio returns
    port_returns = df_backtest['Actual_Return'].dropna().values
    
    # Standardize the returns (mean=0, variance=1)
    mu = np.mean(port_returns)
    sig = np.std(port_returns, ddof=1)
    std_returns = (port_returns - mu) / sig

    dfs_to_test = [3, 4, 5, 6]
    
    fig, axes = plt.subplots(1, len(dfs_to_test), figsize=(18, 4))
    fig.suptitle("QQ Plots: Portfolio Returns vs. Student-t Distributions", y=1.05)
    
    for i, df in enumerate(dfs_to_test):
        # Generate the QQ plot
        stats.probplot(std_returns, dist=stats.t, sparams=(df,), plot=axes[i])
        axes[i].set_title(f"Student-t (df={df})")
        axes[i].get_lines()[0].set_markersize(3)
        axes[i].get_lines()[0].set_alpha(0.6)
        
    plt.tight_layout()
    plt.show()




##################################################################### ------------------------------------------------------------






def test_square_root_of_time_rule(df_backtest, alpha=0.01):
    """
    Computes exact non-overlapping multi-day VaR vs. the Sqrt(T) approximation.
    """
    # Extract daily portfolio returns
    daily_returns = df_backtest['Actual_Return'].dropna()
    
    # 1. Physical Aggregation (Non-overlapping sums)
    # Using integer division (//) on the index to group into buckets of 5 and 10
    ret_5d = daily_returns.groupby(np.arange(len(daily_returns)) // 5).sum()
    ret_10d = daily_returns.groupby(np.arange(len(daily_returns)) // 10).sum()
    
    # 2. Empirical Historical VaR for each timeframe
    var_1d_empirical = -np.percentile(daily_returns, alpha * 100)
    var_5d_empirical = -np.percentile(ret_5d, alpha * 100)
    var_10d_empirical = -np.percentile(ret_10d, alpha * 100)
    
    # 3. Square-Root-of-Time Approximation
    var_5d_sqrt = var_1d_empirical * np.sqrt(5)
    var_10d_sqrt = var_1d_empirical * np.sqrt(10)
    
    # 4. Compile Results
    results = pd.DataFrame({
        'Empirical HS VaR': [var_1d_empirical, var_5d_empirical, var_10d_empirical],
        'Sqrt(T) Rule VaR': [var_1d_empirical, var_5d_sqrt, var_10d_sqrt],
        'Underestimation Error': [0.0, 
                                  var_5d_empirical - var_5d_sqrt, 
                                  var_10d_empirical - var_10d_sqrt]
    }, index=['1-Day', '5-Day', '10-Day'])
    
    # Convert to percentages for readability
    return results * 100





def evaluate_backtest_performance(df_backtest, alpha=0.01):
    """
    Evaluates VaR violations and Expected Shortfall per year.
    Assumes df_backtest contains 'Actual_Return' and pairs of 'VaR_X' and 'ES_X'.
    """
    # Identify the models by looking for 'VaR_' in column names
    models = [col.replace('VaR_', '') for col in df_backtest.columns if 'VaR_' in col]
    
    # 1. Create Violation Indicator Columns - If actual return < -VaR, it's a not violation(ot was an anticipated loss), else it's a violation (unexpected loss)
    for model in models:
        df_backtest[f'Viol_{model}'] = df_backtest['Actual_Return'] < -df_backtest[f'VaR_{model}']
        
    df_backtest['Year'] = df_backtest.index.year
    results = []
    
    # 2. Compute Annual Metrics
    for year, group in df_backtest.groupby('Year'):
        trading_days = len(group)
        expected_violations = trading_days * alpha
        
        year_data = {'Year': year, 'Trading_Days': trading_days, 'Expected_Violations': expected_violations}
        
        for model in models:
            # Actual Violations
            actual_viol = group[f'Viol_{model}'].sum()
            year_data[f'{model}_Actual_Viol'] = actual_viol
            
            # Actual Shortfall (Average loss ON violation days)
            violation_mask = group[f'Viol_{model}'] == True
            if actual_viol > 0:
                actual_shortfall = -group.loc[violation_mask, 'Actual_Return'].mean()
                predicted_es = group.loc[violation_mask, f'ES_{model}'].mean()
            else:
                actual_shortfall = 0.0
                predicted_es = group[f'ES_{model}'].mean() # No violations, just take average ES
                
            year_data[f'{model}_Actual_Shortfall'] = actual_shortfall
            year_data[f'{model}_Predicted_ES'] = predicted_es
            
        results.append(year_data)
        
    df_annual_summary = pd.DataFrame(results).set_index('Year')
    
    return df_backtest, df_annual_summary

print("Backtesting all portfolios")

# Store your weight dataframes in a dictionary
portfolio_weights = {
    'Equal_Weight': df_eq,
    'Risk_Parity': df_rp,
    'RMV_Optimized': df_rmv
}

# Dictionary to hold the final backtest DataFrames
dict_backtests = {}

for port_name, df_w in portfolio_weights.items():
    print(f"\n--- Backtesting {port_name} Portfolio ---")
    
    # Run the parallel engine for this specific set of weights
    df_bt = run_parallel_backtest(
        log_returns, 
        df_w, 
        window=250, 
        alpha=0.01, 
        n_jobs=-1
    )
    
    # Store the result
    dict_backtests[port_name] = df_bt

# =====================================================================
# 1. VISUALIZATION FUNCTION DEFINITION
# =====================================================================


def plot_interactive_violations(df_backtest):
    """
    Graphically investigates the dependence of VaR violations across ALL models interactively.
    """
    fig = go.Figure()

    # 1. Plot the actual portfolio returns (Base Layer)
    fig.add_trace(go.Scatter(
        x=df_backtest.index, y=df_backtest['Actual_Return'],
        mode='lines', name='Actual Return',
        line=dict(color='lightgray', width=1.5), opacity=0.7
    ))

    model_styles = {
        'HS':   {'color': '#1f77b4', 'symbol': 'circle'},      # Blue
        'Norm': {'color': '#2ca02c', 'symbol': 'diamond'},     # Green
        't':    {'color': '#ff7f0e', 'symbol': 'x'}            # Orange
    }

    models = [col.replace('VaR_', '') for col in df_backtest.columns if 'VaR_' in col]

    for model in models:
        color = model_styles.get(model, {}).get('color', 'black')
        symbol = model_styles.get(model, {}).get('symbol', 'circle')

        # Add the VaR Threshold Line
        fig.add_trace(go.Scatter(
            x=df_backtest.index, y=-df_backtest[f'VaR_{model}'],
            mode='lines', name=f'-{model} VaR (99%)',
            line=dict(color=color, width=1.5, dash='solid')
        ))

        # Filter and plot violations
        violation_mask = df_backtest[f'Viol_{model}'] == True
        violations = df_backtest[violation_mask]

        if not violations.empty:
            fig.add_trace(go.Scatter(
                x=violations.index, y=violations['Actual_Return'],
                mode='markers', name=f'{model} Violations',
                marker=dict(color=color, size=8, symbol=symbol, line=dict(width=1, color='black')),
                hovertemplate=f"<b>{model} Violation</b><br>Loss: %{{y:.4f}}<extra></extra>"
            ))

    fig.update_layout(
        title="Interactive VaR Violations Over Time (All Models)",
        xaxis_title="Date", yaxis_title="Daily Return",
        hovermode="x unified", template="plotly_white",
        legend=dict(yanchor="bottom", y=0.01, xanchor="right", x=0.99, bgcolor="rgba(255, 255, 255, 0.8)"),
        xaxis=dict(rangeslider=dict(visible=True), type="date")
    )
    fig.add_hline(y=0, line_width=1, line_color="black")
    fig.show()


# =====================================================================
# 2. FINAL EXECUTION BLOCK: EVALUATE ALL PORTFOLIOS
# =====================================================================
cols_to_display = ['Trading_Days', 'Expected_Violations', 'HS_Actual_Viol', 'Norm_Actual_Viol', 'HS_Actual_Shortfall']

# --- Equal Weight Evaluation ---
df_bt_eq = dict_backtests['Equal_Weight']
df_bt_eq, df_summary_eq = evaluate_backtest_performance(df_bt_eq, alpha=0.01)
try:
    df_summary_eq = append_kupiec_tests(df_summary_eq, alpha=0.01)
except NameError:
    pass
print("\n--- Equal Weight Annual Summary ---")
display(df_summary_eq[cols_to_display])

# --- Risk Parity Evaluation ---
df_bt_rp = dict_backtests['Risk_Parity']
df_bt_rp, df_summary_rp = evaluate_backtest_performance(df_bt_rp, alpha=0.01)
try:
    df_summary_rp = append_kupiec_tests(df_summary_rp, alpha=0.01)
except NameError:
    pass
print("\n--- Risk Parity Annual Summary ---")
display(df_summary_rp[cols_to_display])

# --- RMV Optimized Evaluation ---
df_backtest_rmv = dict_backtests['RMV_Optimized']
df_backtest_rmv, df_summary_rmv = evaluate_backtest_performance(df_backtest_rmv, alpha=0.01)

# --- Execution ---
print("Multi-Day VaR Scaling Analysis (in %):")
df_multi_day = test_square_root_of_time_rule(df_backtest_rmv, alpha=0.01)
display(df_multi_day.round(2))

try:
    df_summary_rmv = append_kupiec_tests(df_summary_rmv, alpha=0.01)
except NameError:
    pass
print("\n--- RMV Optimized Annual Summary ---")
# RMV has t_Actual_Viol and t_Actual_Shortfall, so we display an expanded set
display(df_summary_rmv[['Trading_Days', 'Expected_Violations', 
                        'HS_Actual_Viol', 'Norm_Actual_Viol', 't_Actual_Viol',
                        'HS_Actual_Shortfall', 'Norm_Actual_Shortfall', 't_Actual_Shortfall']].round(2))

# =====================================================================
# 3. VISUALIZATION
# =====================================================================
# Project the Mutated Data onto the Interactive Plot exactly once
plot_interactive_violations(df_backtest_rmv)

In [ ]:
def plot_single_method_backtest(df_backtest, model_name, portfolio_name="Portfolio"):
    """
    Plots actual portfolio returns, VaR threshold, and violations for one model.
    Example model_name: 'HS', 'Norm', 't', 'GARCH', 'EWMA'
    """

    var_col = f'VaR_{model_name}'
    viol_col = f'Viol_{model_name}'

    fig, ax = plt.subplots(figsize=(14, 5))

    ax.plot(df_backtest.index, df_backtest['Actual_Return'],
            label='Actual Return', color='steelblue', alpha=0.6)

    ax.plot(df_backtest.index, -df_backtest[var_col],
            label=f'-{model_name} VaR (99%)', color='darkorange', alpha=0.9)

    df_viol = df_backtest[df_backtest[viol_col]]
    ax.scatter(df_viol.index, df_viol['Actual_Return'],
               color='red', s=20, label='Violations')

    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'{portfolio_name} - {model_name} Backtest')
    ax.set_xlabel('Date')
    ax.set_ylabel('Daily Return')
    ax.legend()
    plt.tight_layout()
    plt.show()

portfolio_backtests = {
    'Equal_Weight': df_bt_eq,
    'Risk_Parity': df_bt_rp,
    'RMV_Optimized': df_backtest_rmv
}

for port_name, df_bt in portfolio_backtests.items():
    for model in ['HS', 'Norm', 't', 'GARCH', 'EWMA']:
        plot_single_method_backtest(df_bt, model, portfolio_name=port_name)